In [4]:
import pandas as pd
import numpy as np
import json

# 1. LOAD THE STARTER DATASET FILES
flights_df = pd.read_excel("air_cote_divoire_starter_dataset.xlsx", sheet_name='Flights')
bookings_df = pd.read_excel("air_cote_divoire_starter_dataset.xlsx", sheet_name='Bookings')
customers_df = pd.read_excel("air_cote_divoire_starter_dataset.xlsx", sheet_name='Customers')


print(f"Loaded {len(flights_df)} flights and {len(bookings_df)} bookings.")


Loaded 480 flights and 11475 bookings.


In [5]:

# =====================================================================
# ENRICHMENT 1: STRUCTURED DATA GENERATION (Fleet Operating Costs)
# =====================================================================
print("Generating Fleet Operating Cost Ledger...")

flights_df['actual_departure'] = pd.to_datetime(flights_df['actual_departure'])
flights_df['actual_arrival'] = pd.to_datetime(flights_df['actual_arrival'])

# 2. Calculate exact Block Time (Duration) as a timedelta object
flights_df['block_time_delta'] = flights_df['actual_arrival'] - flights_df['actual_departure']

# 3. Convert that duration into decimal hours (e.g., 2 hours 30 mins becomes 2.5)
flights_df['actual_block_hours'] = flights_df['block_time_delta'].dt.total_seconds() / 3600.0

# Define operational cost priors based on aircraft type profiles
aircraft_cost_matrix = {
    'A319':    {'fuel_burn_per_hour': 2400, 'crew_hourly_rate': 450,  'maintenance_hourly': 600},
    'A320':    {'fuel_burn_per_hour': 2500, 'crew_hourly_rate': 450,  'maintenance_hourly': 650},
    'A320neo': {'fuel_burn_per_hour': 2100, 'crew_hourly_rate': 480,  'maintenance_hourly': 550},
    'Q400':    {'fuel_burn_per_hour': 1000, 'crew_hourly_rate': 350,  'maintenance_hourly': 400},
    'A330-900':{'fuel_burn_per_hour': 4700, 'crew_hourly_rate': 900,  'maintenance_hourly': 1200}
}

# Map fleet variables back onto our existing flight records using statistical functions
def compute_operating_cost(row):
    ac_type = row['aircraft_type']
    # Default to A320 if aircraft type is unexpected
    rates = aircraft_cost_matrix.get(ac_type, aircraft_cost_matrix['A320'])
    hours = row['actual_block_hours']
    delay_penalty = (row['delay_min'] * 0.15) / 60  # Assuming that each minute of delay burns fuel at 15% of the hourly fuel burn rate

    fuel_cost = rates['fuel_burn_per_hour'] * (hours + delay_penalty) * 0.95  # Assuming 0.95 USD per kg of fuel
    maintenance_cost = hours * rates['maintenance_hourly']
    crew_cost = hours * rates['crew_hourly_rate']

    total_variable_cost = fuel_cost + maintenance_cost + crew_cost

    return pd.Series([round(fuel_cost, 2), round(total_variable_cost, 2)])
# Apply the calculation to every single flight in the dataset
flights_df[['exact_fuel_cost_usd', 'total_operating_cost_usd']] = flights_df.apply(compute_operating_cost, axis=1)

# Preview the mathematically precise output
flights_df[['flight_id', 'aircraft_type', 'actual_block_hours', 'exact_fuel_cost_usd', 'total_operating_cost_usd']].head()

Generating Fleet Operating Cost Ledger...


,flight_id,aircraft_type,actual_block_hours,exact_fuel_cost_usd,total_operating_cost_usd
0,FLT00001,A319,0.750000,2052.00,2839.50
1,FLT00002,A319,1.000000,2280.00,3330.00
2,FLT00003,A320,1.166667,2770.83,4054.17
3,FLT00004,A319,1.083333,2470.00,3607.50
4,FLT00005,A320,1.166667,2770.83,4054.17


In [6]:
# Save the newly enriched structured dataset
flights_df.to_csv("Flights.csv", index=False)
print("✓ Structured Enrichment Completed: 'enriched_flights_with_costs.csv'")

✓ Structured Enrichment Completed: 'enriched_flights_with_costs.csv'


In [ ]:
import re
import time
import requests
import json

# Point to your local Ollama instance
OLLAMA_URL = "http://localhost:11434/api/generate"

# 3. Merge everything into a single master transaction table
master_df = bookings_df.merge(flights_df, on='flight_id').merge(customers_df, on='customer_id')

# 4. Aggregate the data: Build a holistic profile for EVERY customer
print("Aggregating flight histories for all customers...")
customer_profiles = master_df.groupby(['customer_id', 'loyalty_tier', 'customer_segment', 'preferred_channel']).agg(
    total_flights=('flight_id', 'count'),
    delayed_flights=('flight_status', lambda x: (x == 'Delayed').sum()),
    total_delay_min=('delay_min', 'sum'),
    # Get a unique list of routes they fly
    routes_flown=('route_id', lambda x: ', '.join(x.unique())) 
).reset_index()

# For demonstration, we will only process the first 5 customers to avoid long wait times.
# In production, you would run this over the entire customer_profiles dataframe.
synthetic_reviews = []
print(f"\nGenerating aggregate reviews for {len(customer_profiles)} passengers...\n")

# 5. Iterate and Generate using the aggregate profile

def generate_review_with_ollama(profile):
    # Calculate their personal delay rate to help the LLM understand their pain level
    delay_rate = (profile['delayed_flights'] / profile['total_flights']) * 100
    
    # Build the comprehensive prompt
    prompt = f"""
    You are a frequent or occasional flyer leaving a comprehensive review of your overall 
    experience with Air Côte d'Ivoire over the past year.
    
    Here is your exact flight history with the airline:
    - Customer Segment: {profile['customer_segment']}
    - Loyalty Tier: {profile['loyalty_tier']}
    - Total Flights Taken: {profile['total_flights']}
    - Total Flights Delayed: {profile['delayed_flights']} ({delay_rate:.1f}% delay rate)
    - Total Minutes Delayed Across All Flights: {profile['total_delay_min']} minutes
    - Main Routes Flown: {profile['routes_flown']}
    - Booking Channel Preference: {profile['preferred_channel']}
    
    Task:
    Write a 1 to 3 sentence short review reflecting on your ENTIRE history with the airline. 
    - If your delay rate is 0%, praise their reliability.
    - If you fly a lot but had one delay, be forgiving but constructive.
    - If your delay rate is high, express cumulative frustration.
    - Tailor your tone to your loyalty tier (e.g., Gold members expect premium treatment).
    
    Return a JSON object with exactly two keys:
    1. "verbatim_text": The text of your review.
    2. "sentiment_score": A float between -1.0 (extremely negative/angry), 0.0 (neutral/mixed), and +1.0 (extremely positive/loyal).
    Like this:{{"verbatim_text": "...", "sentiment_score": 0.0}}
    """
    
    try:
        payload = {
        "model": "phi4-mini",
        "prompt": prompt,
        "format": "json",
        "stream": False
        }
        
        response = requests.post(OLLAMA_URL, json=payload)
        full_json = response.json()

        if 'response' not in full_json:
            print(f"Unexpected JSON structure: {full_json}")
            return {"verbatim_text": "Invalid API Response", "sentiment_score": 0.0}
        
        raw_text = full_json['response']

        match = re.search(r'\{.*\}', raw_text, re.DOTALL)
        if match:
            return json.loads(match.group(0))
        return {"verbatim_text": "Review failed to generate.", "sentiment_score": 0.0}
    except Exception as e:
        return {"verbatim_text": f"Error: {str(e)}", "sentiment_score": 0.0}
        
        
    except Exception as e:
        print(f"Error generating for {profile['customer_id']}: {e}")

# 4. Execution Loop
print(f"Starting generation for {len(customer_profiles)} customers...")
results = []

for index, profile in customer_profiles.iterrows():
    print(f"Processing [{index+1}/{len(customer_profiles)}] - {profile['customer_id']}")
    
    review_data = generate_review_with_ollama(profile)
    
    # Combine profile data with review data
    record = profile.to_dict()
    record.update(review_data)
    results.append(record)
    
    # Small sleep to keep the Ollama server happy
    time.sleep(0.5)

# 5. Finalize: Save to CSV
final_df = pd.DataFrame(results)
final_df.to_csv("final_synthetic_reviews.csv", index=False)
print("Success! Dataset saved to 'final_synthetic_reviews.csv'")

Aggregating flight histories for all customers...

Generating aggregate reviews for 182 passengers...

Starting generation for 182 customers...
Processing [1/182] - CUST0001
Processing [2/182] - CUST0002
Processing [3/182] - CUST0004
Processing [4/182] - CUST0007
Processing [5/182] - CUST0008
Processing [6/182] - CUST0009
Processing [7/182] - CUST0010
Processing [8/182] - CUST0012
Processing [9/182] - CUST0013
Processing [10/182] - CUST0017
Processing [11/182] - CUST0018
Processing [12/182] - CUST0022
Processing [13/182] - CUST0023
Processing [14/182] - CUST0025
Processing [15/182] - CUST0026
Processing [16/182] - CUST0028
Processing [17/182] - CUST0030
Processing [18/182] - CUST0031
Processing [19/182] - CUST0032
Processing [20/182] - CUST0033
Processing [21/182] - CUST0035
Processing [22/182] - CUST0039
Processing [23/182] - CUST0040
Processing [24/182] - CUST0041
Processing [25/182] - CUST0043
Processing [26/182] - CUST0046
Processing [27/182] - CUST0047
Processing [28/182] - CUST00

In [17]:
final_df['sentiment_score'].mean()

np.float64(-0.571978021978022)

In [2]:
import pandas as pd

# 1. Lire tout le fichier (sheet_name=None renvoie un dictionnaire)
toutes_les_feuilles = pd.read_excel("air_cote_divoire_starter_dataset.xlsx", sheet_name=None)

# 2. Boucler sur chaque feuille pour l'exporter en CSV indépendant
for nom_onglet, df in toutes_les_feuilles.items():
    # Nettoyer le nom du fichier pour éviter les espaces bizarres
    nom_fichier_csv = f"{nom_onglet.replace(' ', '_')}.csv"
    
    # Exporter en CSV
    df.to_csv(nom_fichier_csv, index=False, encoding="utf-8")
    print(f"Feuille '{nom_onglet}' exportée avec succès sous : {nom_fichier_csv}")

Feuille 'README' exportée avec succès sous : README.csv
Feuille 'Airports' exportée avec succès sous : Airports.csv
Feuille 'Routes' exportée avec succès sous : Routes.csv
Feuille 'Customers' exportée avec succès sous : Customers.csv
Feuille 'Flights' exportée avec succès sous : Flights.csv
Feuille 'Bookings' exportée avec succès sous : Bookings.csv
